# פרויקט לימוד מכונה - חלק ב'

במחברת זו נבנה תהליך עבודה מסודר עבור חלק ב' של הפרויקט: טעינת הנתונים, הכנתם לאימון, חלוקה לסט אימון וסט אימות, ובהמשך אימון והשוואה בין מודלים שונים.

בשלב הנוכחי המחברת כוללת את שלד העבודה ואת שלב טעינת והכנת הנתונים בלבד. סעיפי המודלים מופיעים ככותרות להמשך, אך עדיין לא ממומשים.

## 1. ייבוא ספריות והגדרות ראשוניות

נייבא ספריות בסיסיות הדרושות לטעינת הנתונים, בדיקה ראשונית וחלוקה לסט אימון וסט אימות. בהמשך נוסיף ספריות נוספות רק כאשר נגיע לסעיפי המודלים.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


RANDOM_STATE = 42

## 2. טעינת קבצי הנתונים

נטען את קובץ האימון, קובץ המבחן הסופי וקובץ הדוגמה להגשה. בשלב זה קובץ המבחן הסופי משמש רק לבדיקת מבנה, ולא לבחירת מודלים או לכוונון שלהם.

In [ ]:
BASE_DIR = Path.cwd()

TRAIN_FILE = BASE_DIR / "Xy_train.csv"
TEST_FILE = BASE_DIR / "X_test.csv"
EXAMPLE_SUBMISSION_FILE = BASE_DIR / "y test example.xlsx"

required_files = [TRAIN_FILE, TEST_FILE, EXAMPLE_SUBMISSION_FILE]
missing_files = [file_path.name for file_path in required_files if not file_path.exists()]

if missing_files:
    raise FileNotFoundError(f"הקבצים הבאים חסרים בתיקיית העבודה: {missing_files}")

train_raw = pd.read_csv(TRAIN_FILE)
test_raw = pd.read_csv(TEST_FILE)
example_submission = pd.read_excel(EXAMPLE_SUBMISSION_FILE)

display(train_raw.head())
display(test_raw.head())
display(example_submission.head())

## 3. בדיקות מבנה בסיסיות

נבדוק את גודל הקבצים, שמות העמודות, ערכים חסרים וכפילויות. בדיקות אלה נועדו לוודא שאנחנו עובדים עם הקבצים הנכונים לפני שמתחילים לבנות מודלים.

In [ ]:
data_summary = pd.DataFrame(
    {
        "קובץ": ["אימון", "מבחן סופי", "דוגמת הגשה"],
        "מספר שורות": [len(train_raw), len(test_raw), len(example_submission)],
        "מספר עמודות": [train_raw.shape[1], test_raw.shape[1], example_submission.shape[1]],
        "מספר ערכים חסרים": [
            int(train_raw.isna().sum().sum()),
            int(test_raw.isna().sum().sum()),
            int(example_submission.isna().sum().sum()),
        ],
        "מספר שורות כפולות": [
            int(train_raw.duplicated().sum()),
            int(test_raw.duplicated().sum()),
            int(example_submission.duplicated().sum()),
        ],
    }
)

display(data_summary)

In [ ]:
columns_summary = pd.DataFrame(
    {
        "עמודות בקובץ האימון": pd.Series(train_raw.columns),
        "עמודות בקובץ המבחן הסופי": pd.Series(test_raw.columns),
        "עמודות בקובץ הדוגמה": pd.Series(example_submission.columns),
    }
)

display(columns_summary)

In [ ]:
missing_summary = pd.DataFrame(
    {
        "חסרים באימון": train_raw.isna().sum(),
        "חסרים במבחן הסופי": test_raw.isna().sum(),
    }
).fillna(0).astype(int)

display(missing_summary)

## 4. זיהוי עמודת המטרה והפרדת מאפיינים

נזהה את עמודת המטרה מתוך קובץ האימון בפועל. לא נניח מראש את שם העמודה או את צורת הכתיבה שלה. לאחר הזיהוי נפריד בין מאפייני הקלט לבין משתנה המטרה.

In [ ]:
train_columns = list(train_raw.columns)
test_columns = list(test_raw.columns)

columns_only_in_train = [column for column in train_columns if column not in test_columns]

if len(columns_only_in_train) != 1:
    raise ValueError(
        "לא ניתן לזהות באופן חד-משמעי את עמודת המטרה. "
        f"עמודות שמופיעות רק באימון: {columns_only_in_train}"
    )

target_column = columns_only_in_train[0]
feature_columns = [column for column in train_columns if column != target_column]

missing_in_test = [column for column in feature_columns if column not in test_columns]
extra_in_test = [column for column in test_columns if column not in feature_columns]

if missing_in_test or extra_in_test:
    raise ValueError(
        "מבנה עמודות המאפיינים באימון ובמבחן הסופי אינו תואם. "
        f"חסרות במבחן: {missing_in_test}; עודפות במבחן: {extra_in_test}"
    )

X = train_raw[feature_columns].copy()
y = train_raw[target_column].copy()
X_test_final = test_raw[feature_columns].copy()

print(f"עמודת המטרה שזוהתה: {target_column}")
print(f"מספר מאפיינים: {len(feature_columns)}")

In [ ]:
target_distribution = (
    y.value_counts(dropna=False)
    .rename_axis("ערך המטרה")
    .reset_index(name="מספר רשומות")
)
target_distribution["אחוז"] = (target_distribution["מספר רשומות"] / len(y) * 100).round(2)

display(target_distribution)

## 5. ניקוי נתונים לפי החלטות חלק א'

ניישם את החלטות הניקוי שנקבעו לאחר חלק א', בלי לשנות את קבצי המקור. קובץ האימון יכול לאבד רשומה לא תקינה אם יש לכך הצדקה ברורה, אך בקובץ המבחן הסופי לא נמחק רשומות ולא נשנה את הסדר שלהן.

בשלב זה נתקן ערכים בעייתיים וניצור טבלאות עבודה נקיות בזיכרון המחברת. השלמות ערכים התלויות בסט האימון, קידוד וסקיילינג יבוצעו בהמשך בתוך תהליך המידול, כדי להימנע מדליפת מידע מסט האימות או מקובץ המבחן הסופי.

In [ ]:
service_columns = [
    "Inflight wifi service",
    "Departure/Arrival time convenient",
    "Ease of Online booking",
    "Gate location",
    "Food and drink",
    "Seat comfort",
    "On-board service",
    "Leg room service",
    "Baggage handling",
    "Checkin service",
    "Inflight service",
    "Cleanliness",
]

numeric_like_columns = [
    "Age",
    "Flight Distance",
    "Plane colors",
    "Departure Delay in Minutes",
    "Arrival Delay in Minutes",
    *service_columns,
]


def clean_feature_table(features_df, allow_row_removal=False):
    cleaned = features_df.copy()
    summary = {}

    for column in numeric_like_columns:
        if column in cleaned.columns:
            cleaned[column] = pd.to_numeric(cleaned[column], errors="coerce")

    removed_indices = []

    if "Class" in cleaned.columns:
        valid_classes = ["Eco", "Eco Plus", "Business", "Unknown"]
        invalid_class_mask = (~cleaned["Class"].isin(valid_classes)) & cleaned["Class"].notna()
        summary["ערכי Class לא תקינים"] = int(invalid_class_mask.sum())

        if allow_row_removal:
            removed_indices = cleaned.index[invalid_class_mask].tolist()
            cleaned = cleaned.loc[~invalid_class_mask].copy()
        else:
            cleaned.loc[invalid_class_mask, "Class"] = np.nan

        unknown_mask = cleaned["Class"] == "Unknown"
        summary["ערכי Unknown ב-Class שהוחלפו"] = int(unknown_mask.sum())
        cleaned.loc[unknown_mask, "Class"] = "Business"

    if "Gate location" in cleaned.columns:
        invalid_gate_mask = (~cleaned["Gate location"].between(1, 5)) & cleaned["Gate location"].notna()
        summary["ערכי Gate location לא תקינים שהומרו לחסר"] = int(invalid_gate_mask.sum())
        cleaned.loc[invalid_gate_mask, "Gate location"] = np.nan

    if "Age" in cleaned.columns:
        invalid_age_mask = ((cleaned["Age"] < 0) | (cleaned["Age"] > 110)) & cleaned["Age"].notna()
        summary["ערכי Age לא תקינים שהומרו לחסר"] = int(invalid_age_mask.sum())
        cleaned.loc[invalid_age_mask, "Age"] = np.nan

    if "Flight Distance" in cleaned.columns:
        invalid_distance_mask = (cleaned["Flight Distance"] < 0) & cleaned["Flight Distance"].notna()
        summary["ערכי Flight Distance שליליים שהומרו לחסר"] = int(invalid_distance_mask.sum())
        cleaned.loc[invalid_distance_mask, "Flight Distance"] = np.nan

    if "Plane colors" in cleaned.columns:
        cleaned = cleaned.drop(columns=["Plane colors"])
        summary["Plane colors הוסר"] = 1
    else:
        summary["Plane colors הוסר"] = 0

    if "Age" in cleaned.columns:
        cleaned["Age_Category"] = pd.cut(
            cleaned["Age"],
            bins=[0, 12, 18, 65, np.inf],
            labels=["Child", "Teen", "Adult", "Senior"],
            include_lowest=True,
        ).astype("object")

    if "Flight Distance" in cleaned.columns:
        cleaned["Flight_Type"] = pd.cut(
            cleaned["Flight Distance"],
            bins=[0, 1000, 3000, np.inf],
            labels=["Short-Haul", "Medium-Haul", "Long-Haul"],
            include_lowest=True,
        ).astype("object")

    available_service_columns = [column for column in service_columns if column in cleaned.columns]
    if available_service_columns:
        cleaned["Total_Service_Score"] = cleaned[available_service_columns].mean(axis=1)

    return cleaned, removed_indices, summary


X_clean, removed_train_indices, train_cleaning_summary = clean_feature_table(
    X,
    allow_row_removal=True,
)
y_clean = y.drop(index=removed_train_indices).copy()

missing_target_indices = y_clean[y_clean.isna()].index.tolist()
if missing_target_indices:
    X_clean = X_clean.drop(index=missing_target_indices)
    y_clean = y_clean.drop(index=missing_target_indices)

train_cleaning_summary["רשומות עם יעד חסר שהוסרו"] = len(missing_target_indices)

X_test_clean, removed_test_indices, test_cleaning_summary = clean_feature_table(
    X_test_final,
    allow_row_removal=False,
)

assert len(removed_test_indices) == 0
assert len(X_test_clean) == len(test_raw)
assert X_test_clean.index.equals(test_raw.index)

missing_in_clean_test = [column for column in X_clean.columns if column not in X_test_clean.columns]
extra_in_clean_test = [column for column in X_test_clean.columns if column not in X_clean.columns]

if missing_in_clean_test or extra_in_clean_test:
    raise ValueError(
        "מבנה המאפיינים לאחר הניקוי אינו תואם בין אימון למבחן. "
        f"חסרות במבחן: {missing_in_clean_test}; עודפות במבחן: {extra_in_clean_test}"
    )

X_test_clean = X_test_clean[X_clean.columns].copy()

train_clean = X_clean.copy()
train_clean[target_column] = y_clean

X_model = X_clean.copy()
y_model = y_clean.copy()
X_test_final = X_test_clean.copy()

total_removed_train = len(removed_train_indices) + len(missing_target_indices)

cleaning_summary = pd.DataFrame(
    [
        {"קובץ": "אימון", **train_cleaning_summary, "רשומות Class שהוסרו": len(removed_train_indices), "סהכ רשומות שהוסרו": total_removed_train},
        {"קובץ": "מבחן סופי", **test_cleaning_summary, "רשומות Class שהוסרו": len(removed_test_indices), "סהכ רשומות שהוסרו": len(removed_test_indices)},
    ]
).fillna(0)

display(cleaning_summary)
print("הנתונים הנקיים נשמרו בזיכרון המחברת ומוכנים להמשך העבודה.")

### בדיקות לאחר הניקוי

נבדוק שהניקוי יצר שתי טבלאות עבודה תקינות, שקובץ המבחן הסופי שמר על מספר הרשומות והסדר המקורי, ושעמודות המאפיינים זהות בין האימון למבחן.

In [ ]:
post_clean_summary = pd.DataFrame(
    {
        "קובץ": ["אימון נקי", "מבחן סופי נקי"],
        "מספר שורות": [len(train_clean), len(X_test_final)],
        "מספר עמודות": [train_clean.shape[1], X_test_final.shape[1]],
        "מספר ערכים חסרים": [
            int(train_clean.isna().sum().sum()),
            int(X_test_final.isna().sum().sum()),
        ],
    }
)

display(post_clean_summary)

assert target_column in train_clean.columns
assert target_column not in X_test_final.columns
assert list(train_clean.drop(columns=[target_column]).columns) == list(X_test_final.columns)
assert len(X_test_final) == len(test_raw)
assert X_test_final.index.equals(test_raw.index)

print("בדיקות הניקוי עברו בהצלחה: קבצי המקור נשמרו, והטבלאות הנקיות מוכנות להמשך.")

## 6. הכנת נתונים ראשונית לאימון ולאימות

לאחר יצירת קבצי העבודה הנקיים, נשתמש בהם כבסיס להמשך חלק ב'. קובץ המבחן הסופי עדיין נשאר מחוץ לאימון, לבחירת מודלים ולכוונון, והוא ישמש רק בסוף להפקת החיזויים.

In [ ]:
if y_model.isna().any():
    raise ValueError("נמצאו ערכים חסרים בעמודת המטרה לאחר הניקוי. יש לטפל בכך לפני חלוקת הנתונים.")

original_test_index = X_test_final.index.copy()
original_test_length = len(X_test_final)

print("הנתונים הנקיים מוכנים לחלוקה ראשונית לאימון ולאימות.")

## 7. חלוקה לסט אימון וסט אימות

נחלק את קובץ האימון הנקי לסט אימון ולסט אימות. סט האימות יישמר בצד וישמש להערכת ביצועי המודלים על נתונים שלא שימשו לאימון. קובץ המבחן הסופי נשאר מחוץ לתהליך זה.

In [ ]:
from sklearn.model_selection import train_test_split

stratify_target = y_model if y_model.nunique(dropna=False) > 1 else None

X_train, X_valid, y_train, y_valid = train_test_split(
    X_model,
    y_model,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=stratify_target,
)

split_summary = pd.DataFrame(
    {
        "סט": ["אימון", "אימות", "מבחן סופי"],
        "מספר רשומות": [len(X_train), len(X_valid), len(X_test_final)],
        "מספר מאפיינים": [X_train.shape[1], X_valid.shape[1], X_test_final.shape[1]],
    }
)

display(split_summary)

In [ ]:
train_target_distribution = y_train.value_counts(normalize=True, dropna=False).rename("אימון")
valid_target_distribution = y_valid.value_counts(normalize=True, dropna=False).rename("אימות")

split_target_distribution = (
    pd.concat([train_target_distribution, valid_target_distribution], axis=1)
    .fillna(0)
    .mul(100)
    .round(2)
)

display(split_target_distribution)

In [ ]:
assert len(X_test_final) == original_test_length
assert X_test_final.index.equals(original_test_index)

print("בדיקת קובץ המבחן הסופי הסתיימה: מספר השורות והסדר המקורי נשמרו.")

## 8. עצי החלטה

בסעיף זה נבנה בהמשך עץ החלטה מלא, נכוונן היפר-פרמטרים, נציג את העץ הנבחר, ננתח חשיבות משתנים ונעביר רשומת אימות לדוגמה דרך העץ.

## 9. רשתות נוירונים / MLP

בסעיף זה נבנה בהמשך רשת נוירונים, נריץ מודל ברירת מחדל, נכוונן היפר-פרמטרים ונסביר את הקונפיגורציה שנבחרה.

## 10. אשכולות בשיטת K-Means

בסעיף זה נריץ בהמשך K-Means, נבדוק את ההתאמה בין האשכולות למחלקות ונציג גרפים מתאימים להמחשת המבנה שהתקבל.

## 11. מסווג בייסיאני נאיבי

בסעיף זה נאמן בהמשך שני מסווגים בייסיאניים מסוגים שונים, נציג ביצועים ונבדוק הסתברויות למחלקות עבור רשומת אימות אחת.

## 12. השוואה בין מודלים

בסעיף זה נשווה בהמשך בין המודלים המפוקחים וננסח מסקנה לגבי המודל המתאים ביותר למשימת הסיווג.

## 13. המודל הנבחר

בסעיף זה נציג בהמשך את המודל שנבחר להגשה, את הקונפיגורציה שלו ואת מטריצת הבלבול על סט האימות.

## 14. חיזויים סופיים וייצוא קובץ ההגשה

בסעיף זה נאמן בהמשך את המודל הסופי על נתוני האימון המלאים, נחזה את התוויות עבור קובץ המבחן הסופי ונייצא קובץ אקסל בפורמט קובץ הדוגמה.